In [1]:
from pathlib import Path

import numpy as np 
import pandas as pd
import open3d as o3d
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
from plyfile import PlyData

/home/jgajardo/.local/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Pheno4D

In [ ]:
data_folder = Path('../data/Pheno4D/raw/Maize01')
file_paths = sorted(data_folder.glob('*'))

In [ ]:
df = pd.read_csv(file_paths[0], sep=' ', header=None, index_col=False)

In [ ]:
assert len(df.columns) >= 3, f"Expected at least 3 columns, got {len(df.columns)}"

In [ ]:
data = np.loadtxt(file_paths[0])
data

In [ ]:
SCALAR_DIM = 4
points = data[:, :3]
labels = data[:, SCALAR_DIM] 

In [ ]:
unique_labels = np.unique(labels)
color_map = plt.get_cmap("tab20")  # or any other matplotlib colormap
label_to_color = {label: color_map(i / len(unique_labels))[:3] for i, label in enumerate(unique_labels)}
colors = np.array([label_to_color[label] for label in labels])

In [ ]:
fraction = 0.1
N = points.shape[0]
sample_indices = np.random.choice(N, size=int(N * fraction), replace=False)

In [ ]:
labels

In [ ]:
sample_indices = np.where(points[:, 2] > -5)[0]
sample_indices = np.where(labels[:] == 1.)
sample_indices

In [ ]:
points_downsampled = points[sample_indices]
labels_downdownsampled = labels[sample_indices]
colors_downsampled = colors[sample_indices]

In [ ]:
colors.shape

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(points_downsampled[:, 0], points_downsampled[:, 1], points_downsampled[:, 2], c=colors_downsampled, s=1)  # s=1 for small point size

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('Point Cloud Colored by Column')

plt.show()

In [ ]:
plt.hist(points[:, 2], bins=100)
plt.xlim(-10, 1)

In [ ]:
np.median(points[:, 2])

In [ ]:
fig = go.Figure(data=[go.Scatter3d(
    x=points_downsampled[:, 0],
    y=points_downsampled[:, 1],
    z=points_downsampled[:, 2],
    mode='markers',
    marker=dict(
        size=2,
        color=colors_downsampled,  # Can use labels or other scalar for color
        colorscale='Viridis',
        opacity=0.8
    )
)])
fig.show()

In [ ]:
# Create Open3D point cloud
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points_downsampled)

# If you have color data (float RGB values between 0 and 1)
pcd.colors = o3d.utility.Vector3dVector(colors_downsampled)

# Save to PLY
o3d.io.write_point_cloud("../downsampled_point_cloud.ply", pcd)

## BonnBeetClouds3D

In [2]:
base_path = Path('~/public-mnt/Evaluation/Projects/KP0034_jgajardo/data/external/BonnBeetClouds3Dv0').expanduser().absolute()
base_path.exists()

True

In [30]:
data_folder = base_path / 'val'
sample_file = list(data_folder.glob('*'))[0]

In [22]:
open3d_pcd = o3d.io.read_point_cloud(str(sample_file))
pcd_np = np.asarray(open3d_pcd.points)

Open3D doesn't support reading scalar fields from PLY files, so we'll use `plyfile` instead.

Iterate over folder to check the amount of scalar fields (look for leaf tip)

In [43]:
def get_ply_attributes(ply_file_path):
    ply_data = PlyData.read(ply_file_path)
    vertex = ply_data['vertex']
    return vertex.properties

In [50]:
get_ply_attributes(sample_file)

(PlyProperty('x', 'float'),
 PlyProperty('y', 'float'),
 PlyProperty('z', 'float'),
 PlyProperty('nx', 'float'),
 PlyProperty('ny', 'float'),
 PlyProperty('nz', 'float'),
 PlyProperty('red', 'uchar'),
 PlyProperty('green', 'uchar'),
 PlyProperty('blue', 'uchar'),
 PlyProperty('plant_ids', 'int'),
 PlyProperty('confidence', 'uchar'),
 PlyProperty('leaf_ids', 'int'),
 PlyProperty('plot', 'double'))

In [48]:
for file in data_folder.glob('*'):
    print(len(get_ply_attributes(sample_file)))
    

13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13


There is no keypoint_id in the data!